In [ ]:
"""
═══════════════════════════════════════════════════════════════════════════════
MNIST Classification using Complex-Valued Neural Networks with
Post-Training QPSK Weight Quantization
═══════════════════════════════════════════════════════════════════════════════

This project implements a complex-valued neural network for MNIST digit
classification using QPSK-inspired neural computation. Input pixels are
converted into complex QPSK symbols {(±1 ± j)/√2} and processed through
fully complex-valued linear layers with separate real and imaginary weights.
The network is first trained using continuous latent complex weights and
then converted into a deployment-ready QPSK model through post-training
weight quantization. All weights and biases are snapped to valid QPSK
constellation points before inference, producing a fully quantized RF-style
neural network suitable for communication-inspired hardware systems.
═══════════════════════════════════════════════════════════════════════════════
"""

import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

SCALE = 1 / (2 ** 0.5)
QPSK_TOL = 1e-3


# ----------------------------------------------------
# QPSK Utilities
# ----------------------------------------------------

def quantize_to_qpsk(x):
    # Snaps each complex element to the nearest QPSK symbol by taking the
    # sign of real and imaginary parts and scaling by 1/√2. Zero-sign edge
    # case is mapped to +1 to avoid ambiguous constellation points.
    r = torch.sign(x.real)
    i = torch.sign(x.imag)

    r[r == 0] = 1
    i[i == 0] = 1

    return torch.complex(r * SCALE, i * SCALE)


def is_qpsk(x):
    # Checks whether every element of a complex tensor lies on the QPSK
    # constellation (|real| ≈ |imag| ≈ 1/√2) within QPSK_TOL tolerance.
    # Returns True only if both components pass for all elements.
    r = x.real.detach().cpu().float()
    i = x.imag.detach().cpu().float()

    r_ok = torch.all((torch.abs(torch.abs(r) - SCALE) < QPSK_TOL))
    i_ok = torch.all((torch.abs(torch.abs(i) - SCALE) < QPSK_TOL))

    return bool(r_ok and i_ok)


# ----------------------------------------------------
# Complex LayerNorm
# ----------------------------------------------------

class ComplexLayerNorm(nn.Module):
    # Applies independent LayerNorm to real and imaginary parts separately.
    # Keeps normalization within each component so the complex structure
    # is preserved without cross-component coupling.
    def __init__(self, features):
        # Allocates one LayerNorm per component (real and imaginary).
        super().__init__()

        self.ln_r = nn.LayerNorm(features)
        self.ln_i = nn.LayerNorm(features)

    def forward(self, x):
        # Normalizes real and imaginary parts independently and recombines
        # them into a complex tensor.
        return torch.complex(
            self.ln_r(x.real),
            self.ln_i(x.imag)
        )


# ----------------------------------------------------
# Complex Linear Layer (latent float weights)
# ----------------------------------------------------

class ComplexLinear(nn.Module):
    # Complex-valued linear layer with float latent weights. Uses the standard
    # complex matrix multiply: out = (wr + j·wi)(xr + j·xi) + b, expanded into
    # real and imaginary parts. Weights stay float during training and are
    # snapped to QPSK only at deployment via build_qpsk_model.
    def __init__(self, in_features, out_features):
        # Initializes real/imag weight matrices with He scaling and zero biases.
        # Separate real/imag parameters allow independent gradient updates
        # while representing a single complex weight matrix.
        super().__init__()

        std = (2.0 / in_features) ** 0.5

        self.w_real = nn.Parameter(torch.randn(out_features, in_features) * std)
        self.w_imag = nn.Parameter(torch.randn(out_features, in_features) * std)

        self.b_real = nn.Parameter(torch.zeros(out_features))
        self.b_imag = nn.Parameter(torch.zeros(out_features))

    def forward(self, x):
        # Computes complex matrix multiply: real = xr·wr - xi·wi + br,
        # imag = xr·wi + xi·wr + bi, then returns as a complex tensor.
        xr, xi = x.real, x.imag
        wr, wi = self.w_real, self.w_imag

        real = xr @ wr.T - xi @ wi.T + self.b_real
        imag = xr @ wi.T + xi @ wr.T + self.b_imag

        return torch.complex(real, imag)

    def get_complex_weights(self):
        # Returns the weight and bias as complex tensors by combining the
        # stored real/imag parameter pairs. Used by build_qpsk_model to
        # extract weights before QPSK quantization.
        w = torch.complex(self.w_real.data, self.w_imag.data)
        b = torch.complex(self.b_real.data, self.b_imag.data)

        return w, b


# ----------------------------------------------------
# Network
# ----------------------------------------------------

class FullQPSKNet(nn.Module):
    # Three-layer complex MLP for MNIST. Pixels are paired and converted to
    # QPSK symbols as inputs; hidden layers use ComplexLayerNorm; output
    # logits are the magnitudes of complex output neurons. Trained with float
    # latent weights, deployed with QPSK-snapped weights via build_qpsk_model.
    def __init__(self):
        # Builds three ComplexLinear layers (392→512→512→10) and two
        # ComplexLayerNorm layers for the hidden activations.
        super().__init__()

        self.fc1 = ComplexLinear(392, 512)
        self.fc2 = ComplexLinear(512, 512)
        self.fc3 = ComplexLinear(512, 10)

        self.ln1 = ComplexLayerNorm(512)
        self.ln2 = ComplexLayerNorm(512)

    @staticmethod
    def pixels_to_qpsk(x):
        # Flattens the image and pairs consecutive pixels into complex values:
        # even-indexed pixels → real part, odd-indexed → imaginary part.
        # Each component is sign-snapped to ±1/√2 to form QPSK inputs.
        x_flat = x.view(x.size(0), -1)

        r = torch.sign(x_flat[:, 0::2])
        i = torch.sign(x_flat[:, 1::2])

        r[r == 0] = 1
        i[i == 0] = 1

        return torch.complex(r * SCALE, i * SCALE)

    def forward(self, x):
        # Converts pixels to QPSK symbols, passes through two ComplexLinear
        # layers with LayerNorm, then a final linear layer. Returns the
        # magnitude of complex outputs as class logits.
        x_q = self.pixels_to_qpsk(x)

        h1 = self.ln1(self.fc1(x_q))
        h2 = self.ln2(self.fc2(h1))

        out = self.fc3(h2)

        logits = torch.abs(out)

        return logits


# ----------------------------------------------------
# Build QPSK Deployment Model
# ----------------------------------------------------

def build_qpsk_model(model):
    # Creates a fresh FullQPSKNet and copies QPSK-snapped weights from the
    # trained float model into it. Each layer's complex weights and biases
    # are quantized via quantize_to_qpsk before being written into the
    # deployment model's real/imag parameter tensors.
    qpsk_model = FullQPSKNet()

    for name in ["fc1", "fc2", "fc3"]:

        src = getattr(model, name)
        dst = getattr(qpsk_model, name)

        w, b = src.get_complex_weights()

        w_q = quantize_to_qpsk(w)
        b_q = quantize_to_qpsk(b)

        dst.w_real.data = w_q.real
        dst.w_imag.data = w_q.imag
        dst.b_real.data = b_q.real
        dst.b_imag.data = b_q.imag

    return qpsk_model


# ----------------------------------------------------
# Verify QPSK Weights
# ----------------------------------------------------

def verify_qpsk(model):
    # Iterates over all three layers and checks whether weights and biases
    # lie on the QPSK constellation using is_qpsk(). Prints "QPSK" or
    # "NOT QPSK" for each layer's weights and biases.
    print("\nQPSK Weight Check")

    for name in ["fc1", "fc2", "fc3"]:

        layer = getattr(model, name)

        w = torch.complex(layer.w_real.data, layer.w_imag.data)
        b = torch.complex(layer.b_real.data, layer.b_imag.data)

        print(name, "weights:", "QPSK" if is_qpsk(w) else "NOT QPSK")
        print(name, "bias:", "QPSK" if is_qpsk(b) else "NOT QPSK")


# ----------------------------------------------------
# Evaluation
# ----------------------------------------------------

def evaluate(model, loader, device):
    # Runs inference over a full DataLoader in eval mode with gradients off.
    # Returns top-1 accuracy as a percentage. Used for both the float training
    # model and the QPSK deployment model via the same interface.
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)

            preds = logits.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return 100 * correct / total


# ----------------------------------------------------
# Main Training + Deployment Pipeline
# ----------------------------------------------------

def main():
    # Full pipeline: loads MNIST, trains FullQPSKNet for 20 epochs with Adam
    # and CrossEntropyLoss, then builds the QPSK deployment model, verifies
    # all weights are on the constellation, evaluates deployment accuracy,
    # and saves the deployment model to disk.
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("Device:", device)

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    train_set = datasets.MNIST('./data', train=True, download=True, transform=transform)
    test_set = datasets.MNIST('./data', train=False, download=True, transform=transform)

    train_loader = DataLoader(train_set, batch_size=256, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=256)

    model = FullQPSKNet().to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    epochs = 20

    # ---------------- TRAIN ----------------

    for epoch in range(epochs):

        model.train()

        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = model(images)

            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            preds = logits.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * correct / total
        test_acc = evaluate(model, test_loader, device)

        print(f"Epoch {epoch+1}/{epochs} | Train {train_acc:.2f}% | Test {test_acc:.2f}%")

    print("\nTraining Finished")

    # ---------------- BUILD DEPLOYMENT MODEL ----------------

    qpsk_model = build_qpsk_model(model)

    qpsk_model = qpsk_model.to(device)

    verify_qpsk(qpsk_model)

    print("\nRunning inference with broadcast QPSK weights")

    deploy_acc = evaluate(qpsk_model, test_loader, device)

    print("Deployment Accuracy:", deploy_acc)

    torch.save(qpsk_model.state_dict(), "qpsk_model.pth")

    print("\nSaved deployment QPSK model")


if __name__ == "__main__":
    main()